# Donation Impact Calculator

**Business goal:** Compute cost-per-impact-unit ratios from historical data so the
donate page can instantly show donors what their contribution achieves — e.g.
"Could provide 3 months of safe shelter for a girl".

**Pipeline:** Runs daily (or on demand). Reads donations, allocations, and operational
data → computes ratios → writes to `ml_donation_impact_rates` table.

**Output table pattern:** `IsCurrent` / `ScoredAt` / `ModelVersion` (same as other ML tables).

In [ ]:
import pandas as pd
import numpy as np
import datetime as _dt
from pathlib import Path

from db_config import engine, USE_DB, text

print(f"USE_DB = {USE_DB}")

## 1. Load Data

In [ ]:
CSV_DIR = Path('.')

def load_table(table_name: str) -> pd.DataFrame:
    """Load a table from Azure SQL when USE_DB=True, otherwise fall back to CSV."""
    if USE_DB:
        df = pd.read_sql_table(table_name, engine)
        print(f"  {table_name}: {len(df):,} rows (from DB)")
        return df
    csv_path = CSV_DIR / f"{table_name}.csv"
    if not csv_path.exists():
        print(f"  {table_name}: CSV not found at {csv_path} — returning empty DataFrame")
        return pd.DataFrame()
    df = pd.read_csv(csv_path)
    print(f"  {table_name}: {len(df):,} rows (from CSV)")
    return df

print("Loading tables...")
donations     = load_table('donations')
allocations   = load_table('donation_allocations')
metrics       = load_table('safehouse_monthly_metrics')
recordings    = load_table('process_recordings')
education     = load_table('education_records')
health        = load_table('health_wellbeing_records')
visitations   = load_table('home_visitations')
residents     = load_table('residents')

## 2. Prepare Donation Allocations by Program Area

In [ ]:
# Parse amount_allocated to numeric (it may be stored as string)
allocations['amount_allocated'] = pd.to_numeric(allocations['amount_allocated'], errors='coerce')
allocations = allocations.dropna(subset=['amount_allocated', 'program_area'])

total_by_area = allocations.groupby('program_area')['amount_allocated'].sum()
print("\nTotal allocated by program area (PHP):")
print(total_by_area.to_string())
print(f"\nGrand total allocated: ₱{total_by_area.sum():,.2f}")

## 3. Compute Denominators (Units of Impact)

In [ ]:
# Shelter: total bed-months from safehouse monthly metrics
metrics['active_residents'] = pd.to_numeric(metrics.get('active_residents', pd.Series(dtype='float64')), errors='coerce').fillna(0)
total_bed_months = metrics['active_residents'].sum()

# Counseling: total sessions
total_sessions = len(recordings)

# Education: total monthly records (each row = one girl's monthly record)
total_edu_months = len(education)

# Home visitations
total_visitations = len(visitations)

# Health: total monthly records
total_health_months = len(health)

# Residents served (for transport/general)
total_residents = len(residents)

print("\nUnits of impact (denominators):")
print(f"  Bed-months (shelter):     {total_bed_months:,.0f}")
print(f"  Counseling sessions:      {total_sessions:,}")
print(f"  Education record-months:  {total_edu_months:,}")
print(f"  Home visitations:         {total_visitations:,}")
print(f"  Health record-months:     {total_health_months:,}")
print(f"  Residents served:         {total_residents:,}")

## 4. Compute Cost-per-Unit Ratios

In [ ]:
# Map each program_area to its impact denominator and donor-friendly label
PROGRAM_MAPPING = {
    'Operations':  {'units': total_bed_months,   'label': 'month of safe shelter for a girl'},
    'Wellbeing':   {'units': total_sessions,     'label': 'counseling session'},
    'Education':   {'units': total_edu_months,   'label': 'month of educational support'},
    'Outreach':    {'units': total_visitations,  'label': "home visitation to a girl's family"},
    'Maintenance': {'units': total_bed_months,   'label': 'month of safehouse upkeep'},
    'Transport':   {'units': total_residents,    'label': 'transport trip for a resident'},
}

impact_rates = []

for area, allocated in total_by_area.items():
    mapping = PROGRAM_MAPPING.get(area)
    if mapping is None:
        print(f"  ⚠ Unknown program area '{area}' — skipping")
        continue
    if mapping['units'] == 0 or allocated <= 0:
        print(f"  ⚠ {area}: zero units ({mapping['units']}) or zero allocation (₱{allocated:,.2f}) — skipping")
        continue
    
    cost = allocated / mapping['units']
    impact_rates.append({
        'impact_category': area,
        'cost_per_unit':   round(cost, 2),
        'unit_label':      mapping['label'],
        'total_allocated': round(allocated, 2),
        'total_units':     round(mapping['units'], 2),
    })
    print(f"  ✓ {area}: ₱{cost:,.2f} per {mapping['label']}")

rates_df = pd.DataFrame(impact_rates)
print(f"\n{len(rates_df)} impact rates computed.")
rates_df

## 5. Sanity Check — What Would a ₱5,000 Donation Look Like?

In [ ]:
test_amount = 5000
print(f"A ₱{test_amount:,} donation could provide:\n")

for _, row in rates_df.iterrows():
    units = int(test_amount // row['cost_per_unit'])
    if units >= 1:
        print(f"  • {units} {row['unit_label']}")
    else:
        print(f"  • (less than 1 {row['unit_label']} — would not display)")

## 6. Write to Database

In [ ]:
if USE_DB and len(rates_df) > 0:
    _model_version = _dt.date.today().isoformat()
    _now = _dt.datetime.utcnow()

    rates_df['scored_at'] = _now
    rates_df['model_version'] = _model_version
    rates_df['is_current'] = 1

    with engine.begin() as _conn:
        # Mark previous rates as non-current
        _conn.execute(text("UPDATE ml_donation_impact_rates SET is_current = 0 WHERE is_current = 1"))

        # Insert new rates
        rates_df[['impact_category', 'cost_per_unit', 'unit_label',
                   'total_allocated', 'total_units', 'scored_at',
                   'model_version', 'is_current']].to_sql(
            'ml_donation_impact_rates', _conn, if_exists='append', index=False
        )

    print(f"✓ Wrote {len(rates_df)} impact rates to ml_donation_impact_rates (model v{_model_version})")
else:
    print("Skipping DB write — USE_DB is False (no DB_CONNECTION_STRING set)")
    if len(rates_df) == 0:
        print("No rates to write.")